## Imports And Downloads

In [ ]:
!pip install -q langchain langchain-google-genai pydantic python-dotenv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.18 which is incompatible.
langchain-classic 1.0.4 requires langchain-core<2.0.0,>=1.2.31, but you have langchain-core 0.3.63 which is incompatible.
langchain-classic 1.0.4 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.8 which is incompatible.
langchain-community 0.4.1 requires langchain-core<2.0.0,>=1.0.1, but you have langchain-core 0.3.63 which is incompatible.
langchain-groq 1.1.2 requires langchain-core<2.0.0,>=1.2.8, but you have langchain-core 0.3.63 which is incompatible.
langchain-openai 1.1.14 requires langchain-core<2.0.0,>=1.2.31, but you have langchain-core 0.3.63 which is incompatible.
langgraph 1.1.10 requires langchain-core<2,>=1.3.0, but

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
import os
import json

## Info Class

In [ ]:
class PersonInfo(BaseModel):
    name: Optional[str] = Field(default=None, description="The person's full name.")
    email: Optional[str] = Field(default=None, description="The person's email address.")
    Age: Optional[str] = Field(default=None, description="The person's age.")
    Faculty: Optional[str] = Field(default=None, description="The person's faculty, university, or college.")

## Model + Prompt

In [ ]:
def extract_information(text: str):
    # Retrieve the API key
    api_key = os.getenv("GOOGLE_API_KEY", "")

    # Set up the LLM (Temperature 0 for deterministic extraction)
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0, google_api_key=api_key)

    # Set up the Pydantic Output Parser
    parser = PydanticOutputParser(pydantic_object=PersonInfo)

    # Create the Prompt Template
    prompt = PromptTemplate(
        template="Extract the specific structured fields from the unstructured text below.\nIf any field is missing, strictly return null for it.\n\n{format_instructions}\n\nUnstructured Text:\n{text}",
        input_variables=["text"],
        partial_variables={"format_instructions": parser.get_format_instructions()},
    )

    # Build the LangChain pipeline
    chain = prompt | llm | parser

    try:
        # Execute the chain
        output: PersonInfo = chain.invoke({"text": text})

        print("Validation Successful: The output strictly follows the required JSON schema.\n")
        print("--- Extracted JSON Data ---")
        print(output.model_dump_json(indent=2))
        return output

    except Exception as e:
        print("Failed to extract or validate the output.")
        print(f"Error Details: {e}")
        return None


C:\Users\Gaming Store\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.1) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


## Test Cases

In [ ]:
# --- Test Case 1: All fields present ---
sample_text_1 = """
Hello! My name is Merna. I am a 22-year-old student currently studying at the
Faculty of Computer Science. You can contact me at merna.cs@gmail.com for any questions.
"""

print("================== TEST CASE 1 ==================")
print(f"Input Text: {sample_text_1.strip()}\n")
extract_information(sample_text_1)


================== TEST CASE 1 ==================
Input Text: Hello! My name is Merna. I am a 22-year-old student currently studying at the 
Faculty of Computer Science. You can contact me at merna.cs@gmail.com for any questions.



Validation Successful: The output strictly follows the required JSON schema.

--- Extracted JSON Data ---
{
  "name": "Merna",
  "email": "merna.cs@gmail.com",
  "Age": "22",
  "Faculty": "Faculty of Computer Science"
}


PersonInfo(name='Merna', email='merna.cs@gmail.com', Age='22', Faculty='Faculty of Computer Science')

In [ ]:
# --- Test Case 2: Missing fields (Age and Email missing) ---
sample_text_2 = """
I'm Ahmed. I am looking forward to graduating soon from the Faculty of Arts.
It's an absolute pleasure to join this team.
"""

print("\n================== TEST CASE 2 ==================")
print(f"Input Text: {sample_text_2.strip()}\n")
extract_information(sample_text_2)



================== TEST CASE 2 ==================
Input Text: I'm Ahmed. I am looking forward to graduating soon from the Faculty of Arts.
It's an absolute pleasure to join this team.



Validation Successful: The output strictly follows the required JSON schema.

--- Extracted JSON Data ---
{
  "name": "Ahmed",
  "email": null,
  "Age": null,
  "Faculty": "Faculty of Arts"
}


PersonInfo(name='Ahmed', email=None, Age=None, Faculty='Faculty of Arts')